## 0) Connect to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive


<b>Run (shift+enter)</b> the following cell to import packages & modules (instead of creating a virtual environment)

In [ ]:
try:
  import garth
except:
  !pip install garth gpxpy tcxreader
try:
  import gpxpy
except:
  !pip install gpxpy
try:
  import tcxreader
except:
  !pip install tcxreader
try:
  import calplot
except:
  !pip install calplot

import os
import get_garmin as gg

## 1) Download your Garmin activity files

<b>USER INPUT PARAMS</b>: Enter the number of days before today to download your Garmin running and biking activity data for  

Run the next cell to create a folder in this file's directory, named today's date (YYYYMMDD). Function downloads your .gpx & .tcx files for X number of days before today into that folder. Stores out_dir (output directory) object to be used in the following step.

<b>USER INPUT PARAMS</b>: Enter your Garmin login credentials when prompted 



In [ ]:
NUM_DAYS_BEFORE_TODAY = 5

##############################
## do not change below 

out_dir = gg.get_garmin(NUM_DAYS_BEFORE_TODAY, os.getcwd(), [".tcx", ".gpx"])

## 2) Save .gpx & .tcx activity information to .csv 
Run the next cell to parse activity (.gpx & .tcx) files and save data to a .csv file.


In [ ]:
runGPX_df, new_pt_csv = gg.parse_gpx(out_dir)

## parse tcx files
runTCX_df, bikeTCX_df = gg.parse_tcx(out_dir)

### Move files to archive folder

In [12]:
import shutil
archive_dir = os.path.join(os.getcwd(), "archive")
for file in os.listdir(out_dir):
    shutil.move(os.path.join(out_dir, file), os.path.join(archive_dir, file))
    ## delete today's directory
if len(os.listdir(out_dir)) == 0:
    shutil.rmtree(out_dir)

## 3) Data Visualization

If previous cell wasn't run during this runtime session, run the next cell to import your csv of parsed activity files 

In [ ]:
# import os
# import pandas as pd

# G_name = "allGPX_20240728.csv"
# T_name = "runTCX_20240728.csv"

# runGPX_df = pd.read_csv(G_name)
# runTCX_df = pd.read_csv(T_name)

### i) Route Heatmap
Run the next cell to create your route heatmap of your routes

In [ ]:
heatmap = gg.plot_heatmap(runGPX_df, os.path.join(os.getcwd(), "out", "heatmap.html"))
heatmap

### ii) 3D Routes
<b>USER INPUT PARAMS</b>: set smaller bounds  

Run the next cell to create your 3D interactive map of a spatial subset of your routes


In [ ]:
min_lon = -119.98
max_lon = -119.5
min_lat = 34.4
max_lat = 34.6

##############################
## do not change below 

sub = runGPX_df[(runGPX_df['lat'] > min_lat) & (runGPX_df['lat'] < max_lat) & (runGPX_df['lon'] > min_lon) & (runGPX_df['lon'] < max_lon)]
threedee=gg.plot_3d(df = sub, out_fi = os.path.join(os.getcwd(), "out", "route3d.html"))
threedee

### iii) Heatmap Calendar

<b>USER INPUT PARAMS:</b>
statistic (stat) options
- miles/meters
- ascent_m
- minutes

Run the next cell to create a heatmap calendar / github contributions graph of a statistic (distance, ascent, avg speed) of your choice per day  


In [ ]:
stat = "miles"

##############################
## do not change below 

heatmap_fig = gg.cal_heatmap(runTCX_df, stat, 'YlGn', os.path.join(os.getcwd(), "out", stat+'_heatCal.png'))